read in the data

In [1]:
from data_loader import load_split

In [2]:
data_dir = "C:\\Users\\nia_4\\SMU\\8sem\\nlp\\project\\data"

X_train, y_train = load_split(data_dir, "media", "train")
X_val, y_val = load_split(data_dir, "media", "valid")
X_test, y_test = load_split(data_dir, "media", "test")

print(len(X_train), len(X_val), len(X_test))

26590 2356 1300


feature extraction

In [16]:
from feature_extraction_stylometric import extract_features

In [8]:
# show class imbalance in training set
from collections import Counter
print(Counter(y_train))


Counter({2: 10241, 0: 8861, 1: 7488})


In [4]:
import numpy as np

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
# def build_feature_matrix(texts):
#     return np.array([extract_features(t) for t in texts])

# tfidf = TfidfVectorizer(
#     max_features=5000,
#     ngram_range=(1,2),
#     min_df=5
# )

tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=5000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

svd = TruncatedSVD(n_components=300)

X_train_tfidf = svd.fit_transform(X_train_tfidf)
X_val_tfidf = svd.transform(X_val_tfidf)
X_test_tfidf = svd.transform(X_test_tfidf)
#10-11min to run

In [ ]:
def build_handcrafted_matrix(texts):
    return np.array([extract_features(t) for t in texts])

X_train_hand = build_handcrafted_matrix(X_train)
X_val_hand = build_handcrafted_matrix(X_val)
X_test_hand = build_handcrafted_matrix(X_test)
#30min to run

In [23]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

scaler = StandardScaler()

X_train_hand = scaler.fit_transform(X_train_hand)
X_val_hand = scaler.transform(X_val_hand)
X_test_hand = scaler.transform(X_test_hand)

X_train_final = np.hstack([X_train_hand, X_train_tfidf])
X_val_final = np.hstack([X_val_hand, X_val_tfidf])
X_test_final = np.hstack([X_test_hand, X_test_tfidf])

In [24]:
#added tfidf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(
    solver="saga",
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_final, y_train)

val_preds = model.predict(X_val_final)

print(classification_report(y_val, val_preds))
#3min to run

              precision    recall  f1-score   support

           0       0.49      0.14      0.22      1640
           1       0.57      0.39      0.47       618
           2       0.03      0.51      0.06        98

    accuracy                           0.22      2356
   macro avg       0.36      0.35      0.25      2356
weighted avg       0.49      0.22      0.28      2356



In [25]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

sample_weights = np.array([class_weights[y] for y in y_train])

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",   
    random_state=42
)
model.fit(
    X_train_final,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val_final, y_val)],
    verbose=True
)
val_preds = model.predict(X_val_final)

print(classification_report(y_val, val_preds))
#2min to run

[0]	validation_0-mlogloss:1.10623
[1]	validation_0-mlogloss:1.11512
[2]	validation_0-mlogloss:1.12271
[3]	validation_0-mlogloss:1.13225
[4]	validation_0-mlogloss:1.14330
[5]	validation_0-mlogloss:1.15038
[6]	validation_0-mlogloss:1.15925
[7]	validation_0-mlogloss:1.16737
[8]	validation_0-mlogloss:1.17605
[9]	validation_0-mlogloss:1.18569
[10]	validation_0-mlogloss:1.19190
[11]	validation_0-mlogloss:1.20031
[12]	validation_0-mlogloss:1.20786
[13]	validation_0-mlogloss:1.21350
[14]	validation_0-mlogloss:1.22132
[15]	validation_0-mlogloss:1.22795
[16]	validation_0-mlogloss:1.23489
[17]	validation_0-mlogloss:1.24254
[18]	validation_0-mlogloss:1.25151
[19]	validation_0-mlogloss:1.25844
[20]	validation_0-mlogloss:1.26571
[21]	validation_0-mlogloss:1.27405
[22]	validation_0-mlogloss:1.28283
[23]	validation_0-mlogloss:1.28881
[24]	validation_0-mlogloss:1.29598
[25]	validation_0-mlogloss:1.30307
[26]	validation_0-mlogloss:1.31103
[27]	validation_0-mlogloss:1.31641
[28]	validation_0-mlogloss:1.3

old code

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=2000, solver = "saga", random_state=42, verbose=1)
model.fit(X_train_scaled, y_train)

val_preds = model.predict(X_val_scaled)

print(classification_report(y_val, val_preds))

convergence after 62 epochs took 2 seconds
              precision    recall  f1-score   support

           0       0.45      0.09      0.15      1640
           1       0.70      0.30      0.42       618
           2       0.04      0.64      0.07        98

    accuracy                           0.17      2356
   macro avg       0.40      0.34      0.21      2356
weighted avg       0.50      0.17      0.22      2356



First run of the simplest logistic regression model performed with a weighted average F1 of 0.14 using only basic stylistic features (mean sentence length, mean word length, uppercase words, punctiation, and number of words) like those explored in [Linguistic features based framework for automatic fake news detection (2022)](https://www.sciencedirect.com/science/article/pii/S0360835222004697). To try to help the model capture more information from the text, we added additional features combining stylometry with deeper NLP analysis similar to those used by [Stylometric Fake News Detection Based on NLP with Named Entity Recognition (2023)](https://www.mdpi.com/2079-9292/12/17/3676). Once again running the logitisic regression, we see a poerformance F1 score of 0.21, an improvement with the added POS tag frequencies, function words from [Stylometry: Authorship Identification in Web Forums using NLP](https://www.researchgate.net/publication/359950211_Stylometry_Authorship_Identification_in_Web_Forums_using_Natural_Language_Processing), and readability stats. Furthermore, we tried to imrpove the log reg model with the addition of lightweight features such as punctuation diversity, stopword ratio, and sentence complexity that could further capture subtle stylistic choices without significant preprocessing overhead or overlap. We pulled some of these features from older works like [Stylometry: Authorship Identification in Web Forums using NLP, 2021](https://www.researchgate.net/publication/359950211_Stylometry_Authorship_Identification_in_Web_Forums_using_Natural_Language_Processing), which resulted in a slight increase in the weighted average F1 to be 0.22.

In [27]:
import lightgbm as lgb
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, n_jobs=-1)
model.fit(X_train_feat, y_train)
val_preds = model.predict(X_val_feat)

print(classification_report(y_val, val_preds))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009563 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9435
[LightGBM] [Info] Number of data points in the train set: 26590, number of used features: 37
[LightGBM] [Info] Start training from score -1.098876
[LightGBM] [Info] Start training from score -1.267233
[LightGBM] [Info] Start training from score -0.954136
              precision    recall  f1-score   support

           0       0.49      0.10      0.17      1640
           1       0.53      0.27      0.36       618
           2       0.04      0.68      0.07        98

    accuracy                           0.17      2356
   macro avg       0.35      0.35      0.20      2356
weighted avg       0.48      0.17      0.21      2356



c:\Users\nia_4\SMU\8sem\nlp\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## SAVE probabilities

In [ ]:
val_probs = model.predict_proba(X_val_scaled)
test_probs = model.predict_proba(X_test_scaled)  

import numpy as np

np.save("stylometric_val_probs.npy", val_probs)
np.save("stylometric_test_probs.npy", test_probs)
np.save("stylometric_val_labels.npy", y_val)

# Optional but useful
np.save("stylometric_test_labels.npy", y_test)

save other

In [ ]:
import numpy as np
import pandas as pd
import joblib

pd.DataFrame(X_train_feat).to_csv('X_train_features.csv', index=False)
pd.DataFrame(X_val_feat).to_csv('X_val_features.csv', index=False)
pd.DataFrame(y_train, columns=['label']).to_csv('y_train.csv', index=False)
pd.DataFrame(y_val, columns=['label']).to_csv('y_val.csv', index=False)

joblib.dump(model, 'stylometric_model.pkl')


In [ ]:
model = joblib.load('stylometric_model.pkl')
val_preds = model.predict(X_val_feat)